In [ ]:
"""
Confusion matrix (0=TN, 1=FP, 2=FN, 3=TP) as georeferenced GeoTIFFs
+ Per-chip CSV with FP/FN/TP/TN pixel counts for the FP rate per land-use code.
Pre-processing is identical to train.py / compute_test_metrics.py.
"""
import os
import numpy as np
import pandas as pd
import glob, os
import torch
import rasterio
import segmentation_models_pytorch as smp
from pathlib import Path
from torch.utils.data import DataLoader

from train import (
    FlowerstripsDataset, MONTHBANDS, NORM_COEF, CHIP_SIZE,
    NUM_CHANNELS, NUM_WORKERS, MODELFOLDER,
    LABELS_FOLDER, IMAGES_FOLDER, device,
)

# ── CONFIG ──
THRESHOLD  = 0.70   # SWIR Transfer-Modell
ENCODER    = "resnet34"
MODEL_PATH = str(Path(MODELFOLDER) / "FS_unet_resnet34_pw15_bs8_wd0e+00_rgb_nir_swir.pt")

DATASETS = {
    #"BB_test": {
        #"csv":     ".../test_chips_list.csv",
        #"img_dir": IMAGES_FOLDER,
        #"lbl_dir": LABELS_FOLDER,
        #"out_dir": ".../BB/confusion_rasters/",
        #"csv_out": ".../BB/confusion_per_chip_BB_test.csv",
    #},
    "NRW": {
        "img_dir": ".../NRW/SR-Chips/RG_NIR_SWIR",
        "lbl_dir": ".../NRW/Labels_filtered/",
        "out_dir": ".../NRW/confusion_rasters/",
        "csv_out": ".../NRW/confusion_per_chip_NRW.csv",
    },
}

COLORMAP = {
    0: (0, 0, 0, 0),        # TN transparent
    1: (227, 26, 28, 255),  # FP red
    2: (31, 120, 180, 255), # FN blue
    3: (51, 160, 44, 255),  # TP green
}


def build_split_from_dir(img_dir, lbl_dir, ext=".tif"):
    img_files = sorted(glob.glob(os.path.join(img_dir, f"*{ext}")))
    keep = []
    for ip in img_files:
        n = os.path.basename(ip)
        lp = os.path.join(lbl_dir, n)
        if os.path.exists(lp):
            keep.append((ip, lp, n))
    return [k[0] for k in keep], [k[1] for k in keep], [k[2] for k in keep]


def load_model():
    model = smp.Unet(encoder_name=ENCODER, encoder_weights=None,
                     in_channels=NUM_CHANNELS, classes=1, activation=None)
    state = torch.load(MODEL_PATH, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    elif isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state)
    model.to(device).eval()
    return model


def run(model, cfg):
    os.makedirs(cfg["out_dir"], exist_ok=True)
    img_paths, lbl_paths, names = build_split_from_dir(cfg["img_dir"], cfg["lbl_dir"])

    ds = FlowerstripsDataset(img_paths, lbl_paths, MONTHBANDS, NORM_COEF,
                             CHIP_SIZE, augment=False)
    loader = DataLoader(ds, batch_size=1, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    rows = []
    with torch.no_grad():
        for idx, (img, lbl) in enumerate(loader):
            img = img.to(device, non_blocking=True)
            prob = torch.sigmoid(model(img))
            pred = (prob.squeeze() > THRESHOLD).cpu().numpy().astype(np.uint8)
            target = (lbl.squeeze() > 0.5).cpu().numpy().astype(np.uint8)

            # Georeferencing + original dimensions from the Chip GeoTIFF.
            with rasterio.open(img_paths[idx]) as src:
                out_profile = src.profile.copy()
                H, W = src.height, src.width

            # Remove padding (anchored at the top left) -> revert to original H/W.
            pred   = pred[:H, :W]
            target = target[:H, :W]

            fp = (pred == 1) & (target == 0)
            fn = (pred == 0) & (target == 1)
            tp = (pred == 1) & (target == 1)

            conf = np.zeros((H, W), dtype=np.uint8)
            conf[fp] = 1
            conf[fn] = 2
            conf[tp] = 3

            out_profile.update(count=1, dtype="uint8", nodata=0, compress="lzw")
            out_path = os.path.join(cfg["out_dir"], names[idx])
            with rasterio.open(out_path, "w", **out_profile) as dst:
                dst.write(conf, 1)
                dst.write_colormap(1, COLORMAP)

            fp_n = int(fp.sum()); fn_n = int(fn.sum()); tp_n = int(tp.sum())
            n_pix = H * W
            rows.append({
                "chip":           names[idx],
                "raster_path":    out_path,
                "tp": tp_n, "fp": fp_n, "fn": fn_n,
                "tn":             n_pix - tp_n - fp_n - fn_n,
                "n_pixels":       n_pix,
                "label_pos_frac": round(int(target.sum()) / n_pix, 6),
                "pred_pos_frac":  round(int(pred.sum())   / n_pix, 6),
                "has_fp":         bool(fp_n > 0),   
            })

    df = pd.DataFrame(rows)
    df.to_csv(cfg["csv_out"], index=False)
    n_fp_chips = int(df["has_fp"].sum())
    print(f"  Per-Chip CSV: {cfg['csv_out']}  "
          f"({len(df)} Chips, {n_fp_chips} mit FP, "
          f"ΣFP={int(df['fp'].sum())} px)", flush=True)


def main():
    model = load_model()
    for key, cfg in DATASETS.items():
        print(f"--- {key} (t={THRESHOLD}) ---", flush=True)
        run(model, cfg)
        print(f"done -> {cfg['out_dir']}", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
import glob, os
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape

CONF_DIR = ".../NRW/confusion_rasters"
GPKG_OUT = ".../NRW/fp_polygons_NRW_test.gpkg"

geoms, records, crs = [], [], None
for tif in glob.glob(os.path.join(CONF_DIR, "*.tif")):
    if os.path.basename(tif).startswith(("pred_", "prob_")):
        continue 
    with rasterio.open(tif) as src:
        conf = src.read(1)          # 0=TN,1=FP,2=FN,3=TP
        transform = src.transform
        if crs is None:
            crs = src.crs
    fp = (conf == 1).astype("uint8")
    if fp.sum() == 0:
        continue
    for geom, val in shapes(fp, mask=fp.astype(bool),
                            transform=transform, connectivity=8):
        if val != 1:
            continue
        poly = shape(geom)
        geoms.append(poly)
        records.append({"chip": os.path.basename(tif), "area_m2": poly.area})

gdf = gpd.GeoDataFrame(records, geometry=geoms, crs=crs)
gdf.to_file(GPKG_OUT, driver="GPKG")
print(f"{len(gdf)} FP-Polygons -> {GPKG_OUT} (CRS={crs})")

In [ ]:
import glob, os
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape

CONF_DIR = "/workspaces/flowerstrips/ml/NRW/confusion_rasters"
GPKG_OUT = "/workspaces/flowerstrips/ml/NRW/fp_polygons_NRW_test_pos_frac.gpkg"
ONLY_EMPTY_LABEL = True   # True = nur Chips ohne jegliches Ground-Truth-Label

geoms, records, crs = [], [], None
for tif in glob.glob(os.path.join(CONF_DIR, "*.tif")):
    if os.path.basename(tif).startswith(("pred_", "prob_")):
        continue
    with rasterio.open(tif) as src:
        conf = src.read(1)          # 0=TN,1=FP,2=FN,3=TP
        transform = src.transform
        if crs is None:
            crs = src.crs

    label_present = ((conf == 2) | (conf == 3)).any()   # FN oder TP -> Label da
    if ONLY_EMPTY_LABEL and label_present:
        continue

    fp = (conf == 1).astype("uint8")
    if fp.sum() == 0:
        continue
    for geom, val in shapes(fp, mask=fp.astype(bool),
                            transform=transform, connectivity=8):
        if val != 1:
            continue
        poly = shape(geom)
        geoms.append(poly)
        records.append({
            "chip": os.path.basename(tif),
            "area_m2": poly.area,
            "label_empty": not label_present,
        })

gdf = gpd.GeoDataFrame(records, geometry=geoms, crs=crs)
gdf.to_file(GPKG_OUT, driver="GPKG")
print(f"{len(gdf)} FP-Polygone -> {GPKG_OUT} "
      f"({'only empty Chips' if ONLY_EMPTY_LABEL else 'all Chips'}, CRS={crs})")

In [ ]:
"""
Extracts from nrw_per_chip.csv the chips WITHOUT ground-truth labels,
on which the model nevertheless generated false positives.

Definition (consistent with the FP analysis):
  - label_pos_frac == 0  -> Chip contains not a single labelled pixel (unlabelled)
  - fp > 0               -> at least one false-positive pixel predicted

These chips form the basis for the land-use code overlay, as FP pixels
surrounding genuine blue stripes (on labelled chips) are excluded.
"""
import pandas as pd

PER_CHIP_CSV = ".../NRW/nrw_per_chip.csv"
OUT_CSV      = ".../NRW/nrw_fp_unlabelled_chips.csv"

df = pd.read_csv(PER_CHIP_CSV)

# Chips ohne Label, aber mit False Positives
mask = (df["label_pos_frac"] == 0) & (df["fp"] > 0)
fp_chips = df.loc[mask].copy()

# nach Anzahl FP-Pixel absteigend sortieren (die "schlimmsten" oben)
fp_chips = fp_chips.sort_values("fp", ascending=False).reset_index(drop=True)

# nur die relevanten Spalten rausschreiben (chip-Name reicht fuer die Verschneidung,
# fp/pred_pos_frac als Zusatzinfo)
cols = [c for c in ["chip", "fp", "pred_pos_frac", "label_pos_frac"] if c in fp_chips.columns]
fp_chips[cols].to_csv(OUT_CSV, index=False)

# ── Zusammenfassung ───────────────────────────────────────────────────────────
n_total     = len(df)
n_unlabelled = int((df["label_pos_frac"] == 0).sum())
n_fp_chips  = len(fp_chips)
total_fp_px = int(fp_chips["fp"].sum())

print(f"Total chips:                        {n_total}")
print(f"of which unlabelled (label_pos_frac=0): {n_unlabelled}")
print(f"of which with FP (fp>0):                 {n_fp_chips}")
print(f"FP-Pixel-Total for these chips:    {total_fp_px}")
print(f"Chip list saved: {OUT_CSV}")